# Burunov Bot — тест F5-TTS-Russian на Colab T4

**Цель:** проверить что модель hotstone228/F5-TTS-Russian работает, синтезирует русскую речь голосом Бурунова, замерить RTF (real-time factor).

**Что делать:**
1. Загрузи референс аудио Бурунова (5-15 сек чистого голоса) в `/content/burunov_ref.wav`
2. Прогони все ячейки по очереди
3. Если RTF < 1.0 на T4 — модель потянет real-time на G1 (Jetson GPU)
4. Если RTF > 2.0 — на G1 будет медленно, падаем на Piper

**Бесплатно:** Colab T4, ~12 часов/день.

## 1. Установка F5-TTS (2-3 минуты)

In [ ]:
!git clone https://github.com/SWivid/F5-TTS.git /content/F5-TTS
%cd /content/F5-TTS
!pip install -e .
!pip install huggingface_hub soundfile torchaudio

## 2. Проверка GPU

In [ ]:
import torch
print(f'CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')
else:
    print('GPU НЕ доступно! Поменяй runtime: Runtime → Change runtime type → T4 GPU')

## 3. Загрузка модели hotstone228/F5-TTS-Russian

In [ ]:
from huggingface_hub import snapshot_download

print('Скачиваем модель...')
model_path = snapshot_download(repo_id='hotstone228/F5-TTS-Russian')
print(f'Модель скачана в: {model_path}')

import os
print('\nФайлы в репо:')
for f in os.listdir(model_path):
    full = os.path.join(model_path, f)
    size = os.path.getsize(full) / 1e6
    print(f'  {f} ({size:.1f} MB)')

## 4. Загрузка референс аудио Бурунова

**Вариант 1:** Загрузи `burunov_ref.wav` через панель файлов слева (нажми на папку, иконку Upload).

**Вариант 2:** Если у тебя есть URL на аудио, раскомментируй код ниже.

In [ ]:
import os

REF_AUDIO = '/content/burunov_ref.wav'
REF_TEXT = 'Привет, я Сергей Бурунов, и сейчас расскажу вам одну историю.'  # ТОЧНАЯ транскрипция

# Если хочешь скачать по URL:
# !wget -O {REF_AUDIO} 'https://example.com/burunov.wav'

if not os.path.exists(REF_AUDIO):
    print(f'❌ {REF_AUDIO} не найден!')
    print('Загрузи файл через панель слева (Files → Upload)')
else:
    import soundfile as sf
    audio, sr = sf.read(REF_AUDIO)
    print(f'✅ Референс загружен')
    print(f'   Длительность: {len(audio)/sr:.1f} сек')
    print(f'   Частота: {sr} Hz')
    print(f'   Каналов: {audio.shape[1] if len(audio.shape)>1 else 1}')

## 5. Загрузка F5-TTS модели

In [ ]:
import sys
sys.path.insert(0, '/content/F5-TTS')

from f5_tts.api import F5TTS

print('Инициализация F5TTS...')
tts = F5TTS(
    model_type='F5-TTS',
    device='cuda' if torch.cuda.is_available() else 'cpu',
)
print('✅ F5-TTS готова')

## 6. Тест синтеза + замер RTF

RTF (real-time factor) = время_синтеза / длительность_аудио.
- RTF < 1.0 = real-time, можно использовать live
- RTF 1-3 = медленно но терпимо
- RTF > 3 = нельзя использовать на демо (голос запаздывает)

In [ ]:
import time
import soundfile as sf
import numpy as np

TEST_TEXTS = [
    'Угу, щас, Олег Тарасыч... кофеварку найду.',
    'Штирлиц подошёл к окну. Из окна дуло. Штирлиц закрыл окно. Дуло исчезло.',
    'Бля, тут кто-то стоит... дай пройти.',
    'Вот ваш кофе, Олег. Не обожгись, бля.',
]

results = []
for i, text in enumerate(TEST_TEXTS):
    print(f'\n[{i+1}/{len(TEST_TEXTS)}] Синтез: "{text[:60]}..."')
    
    out_path = f'/content/test_{i}.wav'
    
    t0 = time.time()
    try:
        wav, sr, _ = tts.infer(
            ref_file=REF_AUDIO,
            ref_text=REF_TEXT,
            gen_text=text,
            file_wave=out_path,
        )
        synth_time = time.time() - t0
        
        # Длительность сгенерированного аудио
        audio, sr_out = sf.read(out_path)
        audio_dur = len(audio) / sr_out
        rtf = synth_time / audio_dur
        
        print(f'  Синтез: {synth_time:.2f}с | Аудио: {audio_dur:.2f}с | RTF={rtf:.2f}')
        results.append({'text': text, 'synth': synth_time, 'audio': audio_dur, 'rtf': rtf, 'path': out_path})
        
        if rtf > 3:
            print('  ⚠️ RTF > 3 — на демо будет запаздывать')
    except Exception as e:
        print(f'  ❌ Ошибка: {e}')
        import traceback; traceback.print_exc()

## 7. Сводка по RTF

Если средний RTF < 1.0 — F5-TTS можно использовать на G1 (Jetson с GPU).
Если средний RTF 1-2 — можно, но с задержкой 1-2 сек на фразу.
Если средний RTF > 2 — переключаемся на Piper ONNX (edge_tts_server.py).

In [ ]:
if results:
    print('\n=== СВОДКА ===')
    print(f'{"Текст":<40} {"Синтез":<10} {"Аудио":<10} {"RTF":<8}')
    print('-' * 70)
    for r in results:
        text_short = r['text'][:38]
        print(f'{text_short:<40} {r["synth"]:<10.2f} {r["audio"]:<10.2f} {r["rtf"]:<8.2f}')
    avg_rtf = sum(r['rtf'] for r in results) / len(results)
    print(f'\nСредний RTF: {avg_rtf:.2f}')
    if avg_rtf < 1.0:
        print('✅ ОТЛИЧНО — можно использовать на G1')
    elif avg_rtf < 2.0:
        print('🟡 ТЕРПИМО — будет небольшая задержка')
    else:
        print('🔴 ПЛОХО — переключаемся на Piper ONNX')

## 8. Послушать результат

Скачай `test_0.wav` ... `test_3.wav` из панели файлов слева и послушай на ноуте.

In [ ]:
from IPython.display import Audio, display

for r in results:
    print(f'\n>>> {r["text"]}')
    display(Audio(r['path']))

## 9. Сохранить модели для переноса на G1

Если F5-TTS работает хорошо — скачай `model_path` (папка с моделью) на ноут, потом зальём на G1.

In [ ]:
print(f'Путь к модели: {model_path}')
print(f'Размер: {sum(os.path.getsize(os.path.join(model_path,f)) for f in os.listdir(model_path))/1e6:.0f} MB')
print('\nЧтобы скачать — заархивируй:')
!cd /content && tar -czf f5_tts_russian.tar.gz -C $(dirname {model_path}) $(basename {model_path})
print('Готово: /content/f5_tts_russian.tar.gz')
print('Скачай через панель файлов слева')

## 10. Сгенерировать пресеты анекдотов (опционально)

Если хочешь сразу сделать запасные wav на 5 тем анекдотов — раскомментируй код ниже.

In [ ]:
# import os
# os.makedirs('/content/presets', exist_ok=True)
#
# PRESETS = {
#     'shtirlits_intro': 'Ну, значицца, Штирлиц... он вообще почти весь восемьдесят шестой по телевизору был.',
#     'shtirlits_joke': 'Штирлиц подошёл к окну. Из окна дуло. Штирлиц закрыл окно. Дуло исчезло.',
#     'volodka_intro': 'А вот Вовочка... ну, который из школы... хулиган...',
#     'volodka_joke': 'Учительница спрашивает Вовочку: Вовочка, какую оценку тебе поставить? Пять, Марьиванна! Ну, ты ещё посчитай. Ладно, четыре, но не меньше!',
#     'rzhevsky_intro': 'Ну этот... поручик наш... Ржевский, да... к дамам постоянно лез.',
#     'rzhevsky_joke': 'Поручик Ржевский на балу подходит к Наташе Ростовой: Наташа, а Наташа, давайте я вас поцелую! Поручик, я же дама! Ну, тем более!',
#     'new_russians_intro': 'А это ещё тогда, в восемьдесят шестом, кооперативы пошли, эти в малиновых пиджаках.',
#     'new_russians_joke': 'Встречаются два новых русских. Один говорит: Я вчера Запорожец купил! Ну и как? Крылья бабочки, малиновый цвет, всё дела!',
#     'chapaev_intro': 'Ну, Чапаев, понятное дело... к нам из гражданской войны пришёл, но мы его любим.',
#     'chapaev_joke': 'Чапаев спрашивает Петьку: Петька, рубль есть? Нет, Василий Иваныч. А два есть? Тоже нет. Ну вот, а ты говоришь — капиталистическая революция.',
# }
#
# for name, text in PRESETS.items():
#     out = f'/content/presets/{name}.wav'
#     print(f'Синтез: {name}')
#     wav, sr, _ = tts.infer(
#         ref_file=REF_AUDIO,
#         ref_text=REF_TEXT,
#         gen_text=text,
#         file_wave=out,
#     )
#
# !cd /content && tar -czf presets.tar.gz presets/
# print('\nГотово: /content/presets.tar.gz — скачай и распакуй на G1')